In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Check CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


# 1. Generates the dataset and split it into training and validation sets.
def generate_dataset(n_samples, noise_level):
    X, y = make_moons(n_samples=n_samples, noise=noise_level, random_state=42)
    X = StandardScaler().fit_transform(X)

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)

    dataset = TensorDataset(X_tensor, y_tensor)
    
    train_size = int(0.8 * n_samples)  #80% of sample data does to training, while 20% will go to validation/testing set 
    val_size = n_samples - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)        #Batchsize can be changed on this line and the next. Default is 64 to increase speed of the model
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)           #smaller batch sizes will increase training time and resource cost, but can lead to more accurate models depending on factors around your data 

    print(f"Number of training samples: {len(train_loader.dataset)}")
    print(f"Number of validation samples: {len(val_loader.dataset)}")
    
    return train_loader, val_loader


# 2. MLP definition
class MLP(nn.Module):
    def __init__(self, input_size, hidden_units, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.fc2 = nn.Linear(hidden_units, output_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))                        # ReLU can be turned off here uncomment the next line and comment out this line to turn off ReLU - Will increase speed but reduce accuracy of model 
       # x = self.fc1(x) 
        x = self.fc2(x)
        return x


def train(model, loader, criterion, optimizer):         #Trainer model 
    model.train()
    total_loss = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()                       
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
    return total_loss / len(loader.dataset)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item() * data.size(0)
    return total_loss / len(loader.dataset)

def compute_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)  # Get class with highest score
            correct += (predicted == target).sum().item()
            total += target.size(0)
    
    return correct / total

                                           #Sets different Hyperparameters to be tested 
noise_levels = [.1, .2, .3]                #Change noise level to change complexity of dataset. Higher noise levels will be harder for models to fit
n_samples_list = [100, 1000, 10000]        #Change number of samples that will be generated for data set. Larger data sets cost more to train, but can help reduce over fitting
hidden_units_list = [4, 16, 64, 256]       #Changes number of neurons in hidden layer. More neurons will generally give more accruacy faster, but will increase cost and can cause overfitting later. 

input_size = 2   #input of x,y
output_size = 2  # output of moon 1 or 2 

for noise in noise_levels:
    fig, axs = plt.subplots(3, 4, figsize=(20, 15))
    fig.suptitle(f"Training and Validation Loss Curves for Noise Level {noise}", fontsize=14)

    print(f'Noise Level: {noise}')
    
    for i, n_samples in enumerate(n_samples_list):
        train_loader, val_loader = generate_dataset(n_samples, noise)
        print(f'n_samples: {n_samples}')
        
        for j, hidden_units in enumerate(hidden_units_list):
            model = MLP(input_size, hidden_units, output_size).to(device)  # <-- Moved to GPU

            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.01)

            train_losses, val_losses = [], []
            train_accuracies, val_accuracies = [], []

            print(f'Hidden Units: {hidden_units}')

            for epoch in range(50):
                train_loss = train(model, train_loader, criterion, optimizer)
                val_loss = validate(model, val_loader, criterion)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                train_acc = compute_accuracy(model, train_loader)
                val_acc = compute_accuracy(model, val_loader)

                train_accuracies.append(train_acc)
                val_accuracies.append(val_acc)
                print(f'Epoch {epoch+1}/50, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Train Acc: {train_acc:.2%}, Val Acc: {val_acc:.2%}')
                #print(f'Epoch {epoch+1}/100, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

            ax = axs[i, j]
            ax.plot(range(1, 51), train_losses, label='Train Loss', color='gold')
            ax.plot(range(1, 51), val_losses, label='Val Loss', color='purple')
            ax.plot(range(1, 51), train_accuracies, label='Train Acc', color='green')
            ax.plot(range(1, 51), val_accuracies, label='Val Acc', color='blue')
            ax.set_title(f'Samples: {n_samples}, Hidden Units: {hidden_units}')
            ax.set_xlabel('Epochs')
            ax.set_ylabel('Loss')
            ax.legend()

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    print('\n')


# Task 2 — L2 Regularization    #test model with L2 regularization active
n_samples = 1000                # change samples here
noise_level = .2                # change noise level here
lambdas = [1e-4, 1e-3, 1e-2]    #Change L2 lambdas values here 

print(f'Noise Level: {noise_level}')
print(f'n_samples: {n_samples}')

fig, axs = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle(f"Training and Validation Loss with L2 Regularization (n_samples={n_samples}, noise={noise_level})", fontsize=14)

for i, lambda_val in enumerate(lambdas):
    train_loader, val_loader = generate_dataset(n_samples, noise_level)
    print(f'Weight decay: {lambda_val}')

    for j, hidden_units in enumerate(hidden_units_list):
        model = MLP(input_size, hidden_units, output_size).to(device)  # <-- Moved to GPU

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=lambda_val)

        train_losses, val_losses = [], []
        train_accuracies, val_accuracies = [], []

        print(f'Hidden Units: {hidden_units}')

        for epoch in range(50):
            train_loss = train(model, train_loader, criterion, optimizer)
            val_loss = validate(model, val_loader, criterion)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_acc = compute_accuracy(model, train_loader)
            val_acc = compute_accuracy(model, val_loader)

            train_accuracies.append(train_acc)
            val_accuracies.append(val_acc)
            print(f'Epoch {epoch+1}/50, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Train Acc: {train_acc:.2%}, Val Acc: {val_acc:.2%}')
            #print(f'Epoch {epoch+1}/100, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

        ax = axs[i, j]
        ax.plot(range(1, 51), train_losses, label='Train Loss', color='gold')
        ax.plot(range(1, 51), val_losses, label='Val Loss', color='purple')
        ax.plot(range(1, 51), train_accuracies, label='Train Acc', color='green')
        ax.plot(range(1, 51), val_accuracies, label='Val Acc', color='blue')
        ax.set_title(f'λ = {lambda_val}, Hidden Units = {hidden_units}')
        ax.set_xlabel('Epochs')
        ax.set_ylabel('Loss')
        ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()